# # **키친솔루션 KAMP 머신러닝 강의2 소스 코드2**

## *8장 다중회귀와 분류*
---
* 출판사 : 생능 출판사( http://www.booksr.co.kr/ )
* 저서 : 으뜸 데이터 분석과 머신러닝"(개정판)
* 저자 : 박동규, 강영민, 김병욱
* 책의 소스코드 저장소 : https://github.com/dongupak/DataML
* 안내 : 이 노트북은 책의 코드를 담고 있습니다.
* 저작권 : 본 주피터 노트북 코드는 자유롭게 배포가능하지만 위의 출판사, 저서, 저자표기와 함께 배포해 주십시오.


### 8.3 기대수명 데이터 둘러보기

In [ ]:
import pandas as pd

path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path+'life_expectancy.csv'

life = pd.read_csv(file)
print(life.head(3))

In [ ]:
print('기대수명 데이터의 개요:')
print(life.describe())

In [ ]:
print('life 데이터의 컬럼들')
print(life.columns)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 시본 라이브러리의 크기를 지정
sns.set(rc={'figure.figsize':(22,20)})
# 상관행렬을 나타내어보자. 이때 수치 데이터에 대한 상관계수를 구하도록 하자
correlation_matrix = life.corr(numeric_only=True).round(2)
sns.heatmap(data=correlation_matrix, annot=True)

### 8.4 기대수명과 상관도가 높은 데이터는 무엇인가

In [ ]:
print("기대수명과의 상관계수")
print(life.corr(numeric_only=True).round(3)['Life expectancy'])

In [ ]:
import numpy as np

# 'Life expectancy'와 다른 속성들의 소수점 둘째 자리 상관계수를 구함
c = life.corr(numeric_only=True).round(2)['Life expectancy']
c = np.abs(c)       # 상관계수의 절대값을 취한다
print(c.sort_values(ascending=False)[1:8]) # 상위 7개 값을 출력하자

In [ ]:
print('Life expectancy와 상관도 순서로 정렬 후 하위 5개 값')
print(c.sort_values(ascending=False)[-6:-1]) # 하위 5개 슬라이싱

### 8.5 쌍그래프로 특성 분석하기

In [ ]:
sns.pairplot(life[['Life expectancy', 'Schooling',\
                  'Income composition of resources',\
                  'Adult mortality', 'BMI', 'HIV/AIDS']])
plt.show()

In [ ]:
life.isna().sum().sort_values(ascending=False)

In [ ]:
main_features = ['Schooling',\
                 'Income composition of resources',\
                 'Adult mortality', 'BMI', 'HIV/AIDS']
life[main_features].isna().sum()

In [ ]:
life.dropna(inplace=True) # 결측값 제거

X = life[main_features]
y = life['Life expectancy']
print(X.shape, y.shape ) # 결측값 제거 후 X, y 차원크기

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y,\
                                                    random_state=42,\
                                                    test_size = 0.2)
regr = LinearRegression()
regr.fit(X_train, y_train)
print('선형 회귀 모델의 점수=', f"{regr.score(X_test, y_test):.3f}")

y_pred = regr.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"선형회귀 모델 성능:")
print(f"R2   : {r2:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"MAE  : {mae:.3f}")



### 책에 없음 : 선형회귀 그래프를 시각적으로 확인하는 코드

In [ ]:
plt.figure(figsize=(6,6))
y_test_predict = regr.predict(X_test)
plt.scatter(y_test, y_test_predict)
plt.plot(y_test, y_test, color='r', linewidth=3)

### 8.6 훈련 데이터, 검증 데이터 그리고 특성

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ===============================
# 데이터 준비
# ===============================
path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path + 'life_expectancy.csv'
life = pd.read_csv(file)

life.dropna(inplace=True)  # 결측값 제거

# 'Country', 'Year', 'Status', 'Life expectancy'를 제외한 속성 사용
X = life[['Year','Adult mortality','Infant deaths', 'Alcohol',
          'Percentage expenditure','Hepatitis B', 'Measles',
          'BMI', 'Under-five deaths', 'Polio',
          'Total expenditure','Diphtheria', 'HIV/AIDS', 'GDP',
          'Population','Thinness 1-19 years','Thinness 5-9 years',
          'Income composition of resources', 'Schooling']]
y = life['Life expectancy']

# ===============================
# Train/Test Split + 모델 학습
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2
)

regr = LinearRegression()
regr.fit(X_train, y_train)

# ===============================
# 예측 + 평가 지표
# ===============================
y_pred = regr.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"선형회귀 모델 성능:")
print(f"R2   : {r2:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"MAE  : {mae:.3f}")


### LAB 8-1 : 상관도를 시각화하여 중요 특성을 분석하자.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path+'life_expectancy.csv'

life = pd.read_csv(file)
life.dropna(inplace=True) # 결측값 제거
# 'Country', 'Status', 'Life expectancy'를 제외한 속성 사용
X = life[['Year','Adult mortality','Infant deaths', 'Alcohol',\
         'Percentage expenditure','Hepatitis B', 'Measles',\
         'BMI', 'Under-five deaths', 'Polio',\
         'Total expenditure','Diphtheria', 'HIV/AIDS', 'GDP',\
         'Population','Thinness 1-19 years','Thinness 5-9 years',\
         'Income composition of resources', 'Schooling']]

# 'Life expectancy'와 다른 속성들의 상관계수를 구함
c = life.corr(numeric_only=True)['Life expectancy']
c = np.abs(c)       # 상관계수의 절대값을 취한다
c.sort_values(ascending=True, inplace=True)
c = c[:-1] # 제일 뒤에 있는 Life expectancy는 제거

plt.figure(figsize=(7,7))
plt.barh(c.index, c.values)
plt.title('Correlation Coef with Life expectancy')
plt.xlabel('Features')
plt.ylabel('Correlation Coef')

### 8.7 데이터의 분포가 직선이 아니라면? 다항 회귀 모델을 사용하자

#### 다항회귀 테스트를 위하여 2차 방정식으로 비선형 데이터를 임의로 만들

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

m = 100    # 생성할 데이터의 개수
# 평균값이 0이고 -4에서 4사이에 분포하는 랜덤 값 X
np.random.seed(84)   # 매번 동일한 결과를 얻기 위하여 시드를 84로 부여하자
X = 8 * np.random.rand(m, 1) - 4
# x^2항의 계수가 0.5, x항의 계수가 2, 상수항의 계수가 1
y = 0.5 * X ** 2 + 2 * X + 1 + np.random.randn(m, 1)

plt.figure(figsize=(6,4))
plt.plot(X, y, "b^")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

regr = LinearRegression()
regr.fit(X, y)
print('선형회귀 모델의 점수 =', f"{regr.score(X, y):.3f}")
print('선형회귀 모델의 계수와 절편 =', regr.coef_, regr.intercept_)

In [ ]:
plt.figure(figsize=(7,6))
y_predict = regr.predict(X)
plt.scatter(X, y)
plt.plot(X.flatten(), y_predict, color='r')

### 8.8 다항 회귀 모델을 사용하자

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

t = np.arange(6).reshape(3, -1)   # 간단한 샘플 데이터 t를 생성(2차원)
print(t)

In [ ]:
poly = PolynomialFeatures(degree=2)       # 디폴트 degree=2임
new_t = poly.fit_transform(t)
print(new_t)        # t에 대하여 다항 특성을 추가하고 이를 출력해 보자

In [ ]:
print(poly.get_feature_names_out())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# 8.7 절의 데이터 X
m = 100    # 생성할 데이터의 갯수
# 평균값이 0이고 -4에서 4사이에 분포하는 랜덤 값 X
X = 8 * np.random.rand(m, 1) - 4
# x^2항의 계수가 0.5, x항의 계수가 2, 상수항의 계수가 1
y = 0.5 * X ** 2 + 2 * X + 1 + np.random.randn(m, 1)

poly_features = PolynomialFeatures(degree = 2,\
                                   include_bias = False)
X_poly = poly_features.fit_transform(X)
regr = LinearRegression()
regr.fit(X_poly, y)
print('다항 회귀 모델의 점수 =', f"{regr.score(X_poly, y):.3f}")
print('다항 회귀 모델의 계수 =', regr.coef_, '절편 =', regr.intercept_)

In [ ]:
# -4에서 4사이의 데이터를 생성하자.
domain = np.linspace(-4, 4, 50).reshape(-1, 1)
# domain 데이터에 2차 다항 특성을 추가한 domain_2를 만들자
domain_2 = poly_features.fit_transform(domain)
plt.figure(figsize=(6,4))
y_predict = regr.predict(domain_2)
plt.scatter(X, y)
plt.plot(domain, y_predict, color='r', linewidth=4)

### LAB 8-2 : 3차 방정식을 만들고 다항 회귀 모델을 이용하여 예측하기

In [ ]:
# 2개의 변곡점을 가진 데이터를 생성하고 시각화하는 코드
import numpy as np
import matplotlib.pyplot as plt

m = 100
X = 8 * np.random.rand(m, 1) - 4
y = .5 * X ** 3 + .5 * X ** 2 + X + 3 + np.random.randn(m, 1)
plt.figure(figsize=(6,5))
plt.plot(X, y, "b^")

In [ ]:
# 다항 회귀 모델을 만들고 점수와 계수, 절편을 출력하는 코드
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

poly_features = PolynomialFeatures(degree = 3,\
                                   include_bias = False)
X_poly = poly_features.fit_transform(X)
regr = LinearRegression()
regr.fit(X_poly, y)

print('다항 회귀 모델의 점수 =', f"{regr.score(X_poly, y):.3f}")
print('다항 회귀 모델의 계수 =', regr.coef_)
print('절편 =', regr.intercept_)

# 다항 회귀 곡선을 생성하는 기능
X_new = np.linspace(-4, 4, 100).reshape(100, 1)
X_new_poly = poly_features.transform(X_new)
y_new = regr.predict(X_new_poly)

plt.figure(figsize=(6,6))
plt.plot(X, y, "b^", label='Data')
plt.plot(X_new, y_new, "r-", label="Prediction")
plt.legend()

### 8.9 과대 적합에 주의하자

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

m = 20      # 20개의 데이터를 생성하자
X = 8 * np.random.rand(m, 1) - 4
y = .5 * X ** 3 + .5 * X ** 2 + X + 3 + np.random.randn(m, 1)

# 20개의 데이터를 학습용 데이터, 테스트 데이터로 구분하자
X_train, X_test, y_train, y_test = train_test_split(X, y)

plt.figure(figsize=(6,5))
plt.xlim(-5, 5); plt.ylim(-28, 50) # x, y 범위를 제한함
plt.plot(X_train, y_train, "b^")   # 학습용 데이터를 그리자

In [ ]:
# 다항 회귀 모델을 만들고 점수와 계수, 절편을 출력
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# 다항 특성이 20개가 되도록 하자
poly_features = PolynomialFeatures(degree = 20,\
                                   include_bias = False)
X_poly = poly_features.fit_transform(X_train)
regr = LinearRegression()
regr.fit(X_poly, y_train)

print('다항 회귀 모델의 점수 =', f"{regr.score(X_poly, y_train):.3f}")
print('다항 회귀 모델의 계수 =', regr.coef_)
print('절편 =', regr.intercept_)

In [ ]:
# 테스트 데이터를 다항 회귀 모델의 입력으로 주고 예측 점수를 구해보자
X_poly = poly_features.fit_transform(X_test)

print('테스트 데이터에 대한 모델의 점수 =', regr.score(X_poly, y_test))

In [ ]:
# 다항 회귀 곡선을 생성하는 기능
X_new = np.linspace(-4, 4, 100).reshape(100, 1)
X_new_poly = poly_features.transform(X_new)
y_new = regr.predict(X_new_poly)

plt.figure(figsize=(7,6))
plt.xlim(-5, 5); plt.ylim(-28, 50) # x, y 범위를 제한함

# 학습용 데이터와 다항 회귀 곡선을 그려보고 테스트 데이터도 그려보자
plt.plot(X_train, y_train, 'b^')
plt.scatter(X_test, y_test, c='red', s=200, alpha=0.4)
plt.plot(X_new, y_new, "r-", label="Prediction curve")
plt.legend()

### 8.10 과소 적합, 과대 적합, 그리고 규제

### 8.11 최적화와 규제항을 적용한 릿지 회귀

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# 20개의 데이터를 생성하자
m = 20
X = 8 * np.random.rand(m, 1) - 4
y = .5 * X ** 3 + .5 * X ** 2 + X + 3 + np.random.randn(m, 1)
# 20개의 데이터를 학습용 데이터, 테스트 데이터로 구분하자
X_train, X_test, y_train, y_test = train_test_split(X, y)
# 다항 특성이 20개가 되도록 하자
poly_features = PolynomialFeatures(degree = 20,\
                                  include_bias = False)
X_train_pl = poly_features.fit_transform(X_train)
X_test_pl = poly_features.fit_transform(X_test)

regr = LinearRegression()
regr.fit(X_train_pl, y_train)

print('훈련 데이터의 점수 =', f"{regr.score(X_train_pl, y_train):.3f}")
print('테스트 데이터의 점수 =', f"{regr.score(X_test_pl, y_test):.3f}")

### 8.12 릿지 회귀와 alpha 값

In [ ]:
### 규제를 추가해 보자 ##
from sklearn.preprocessing import StandardScaler

ss = StandardScaler() # 규제가 있는 모델은 스케일러로 변환을 해야 함
ss.fit(X_train_pl)
train_scaled = ss.transform(X_train_pl)
test_scaled = ss.transform(X_test_pl)
### 릿지 회귀 ###
from sklearn.linear_model import Ridge
ridge = Ridge(alpha = 10)
ridge.fit(train_scaled, y_train)
print('훈련 데이터의 점수 =', ridge.score(train_scaled, y_train))
print('테스트 데이터의 점수 =', ridge.score(test_scaled, y_test))

In [ ]:
import matplotlib.pyplot as plt

train_scores = []
test_scores = []
alpha_values = [0.001, 0.01, 1, 10, 100, 1000]
for a in alpha_values:
  ridge = Ridge(alpha = a)
  ridge.fit(train_scaled, y_train)
  train_scores.append(ridge.score(train_scaled, y_train))
  test_scores.append(ridge.score(test_scaled, y_test))

plt.plot(np.log10(alpha_values), train_scores, 'r-', label='Train score')
plt.plot(np.log10(alpha_values), test_scores, 'b:', label='Test score')
plt.xlabel('Alpha value : 10^x scaled')
plt.ylabel('Ridge Regression')
plt.legend(loc="lower left")

# 업그레이드 진행

### dropna보다는 median값으로 보충하고자 판단

#### 결측치 개수와 총 데이터 개수 비교

In [ ]:
import pandas as pd

# 데이터 불러오기
path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path + 'life_expectancy.csv'
df = pd.read_csv(file)

# 전체 행 수
total = len(df)

# 각 컬럼별 결측치 요약
missing_summary = (
    df.isna().sum()
    .to_frame(name='Missing_Count')
    .assign(Total=total)
    .assign(Missing_Ratio=lambda d: (d['Missing_Count'] / d['Total'] * 100).round(2))
)

# 타깃 변수 지정
target = "Life expectancy"

# 상관계수 계산 (수치형만)
corr = df.corr(numeric_only=True)

# 결측치 요약표에 상관계수 컬럼 추가
if target in corr.columns:
    missing_summary['Correlation_with_' + target] = corr[target]
else:
    missing_summary['Correlation_with_' + target] = None

# 정렬: 결측 비율 높은 순
missing_summary = missing_summary.sort_values('Missing_Ratio', ascending=False)

print(missing_summary)


#### Population, Hepatitis B, Total expenditure 은 높은 결측 비율에 비해 낮은 상관도이기 때문에 제거
#### GDP, Alcohol, Income composition of resources, Schooling 은 결측 비율에 비해 높은 상관도이기 때문에 제거 X

### 기존 코드에서 제거했던 컬럼 unique 값 출력

In [ ]:
import pandas as pd

path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path + 'life_expectancy.csv'
df = pd.read_csv(file)

# unique 값 출력
print("=== Country Unique 값 수 & 샘플 ===")
print(len(df['Country'].unique()), "개")
print(df['Country'].unique()[:10])  # 앞 10개만 샘플 출력

print("\n=== Year Unique 값 ===")
print(sorted(df['Year'].unique()))

print("\n=== Status Unique 값 ===")
print(df['Status'].unique())


#### Country와 Year은 제거해도 무방해보임

### Status에 대해 인코딩 후 상관도 수치 출력

In [ ]:
import pandas as pd

path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path + 'life_expectancy.csv'
df = pd.read_csv(file)

# Status를 0/1로 변환
df['Status_Label'] = df['Status'].map({'Developing': 0, 'Developed': 1})

# 상관계수만 출력
corr_val = df[['Status_Label', 'Life expectancy']].corr().iloc[0,1]
print("Status_Label ↔ Life expectancy 상관계수:", corr_val)


#### 높은 상관도 확인, 제거하지 않기로 결정
#### 총 제거 컬럼 : Population, Hepatitis B, Total expenditure, Country, Year

In [ ]:
import pandas as pd

# 데이터 불러오기
path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path + 'life_expectancy.csv'
df = pd.read_csv(file)

# 1) 사용하지 않을 컬럼 제거
drop_cols = ['Population', 'Hepatitis B', 'Total expenditure', 'Country', 'Year']
df = df.drop(columns=drop_cols)

# 2) 모든 결측치를 중앙값으로 보간
df = df.fillna(df.median(numeric_only=True))

# 결과 확인
print(df.isna().sum().sum())   # 남은 결측치 총합 (0이면 성공)

### 특성 개수 선택을 위해 Ridge / Lasso 계수 확인

In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

# 1) 데이터 불러오기
path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path + 'life_expectancy.csv'
df = pd.read_csv(file)

# 컬럼명 공백 제거
df.columns = df.columns.str.strip()

# 2) 사용하지 않기로 한 컬럼 제거
drop_cols = ['Population', 'Hepatitis B', 'Total expenditure', 'Country', 'Year']
df = df.drop(columns=drop_cols)

# 3) Status를 수치화 (Developing=0, Developed=1)
df['Status'] = df['Status'].map({'Developing':0, 'Developed':1})

# 4) 결측치 중앙값 보간
df = df.fillna(df.median(numeric_only=True))

# 5) 타깃과 입력 분리
y = df['Life expectancy']
X = df.drop(columns=['Life expectancy'])

# 6) RidgeCV와 LassoCV 모델 정의
alphas = np.logspace(-3, 3, 50)  # 규제강도 후보
ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RidgeCV(alphas=alphas, cv=5))
])

lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LassoCV(alphas=alphas, cv=5, max_iter=10000, random_state=42))
])

# 7) 교차검증 성능 비교
for name, model in [("Ridge", ridge), ("Lasso", lasso)]:
    scores = cross_val_score(model, X, y, cv=5, scoring="r2")
    print(f"{name} CV R² mean={scores.mean():.3f} std={scores.std():.3f}")

# 8) 계수 안정성 확인 (훈련셋 전체에 피팅)
ridge.fit(X, y)
lasso.fit(X, y)

ridge_coef = ridge.named_steps['model'].coef_
lasso_coef = lasso.named_steps['model'].coef_

coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Ridge_Coef": ridge_coef,
    "Lasso_Coef": lasso_coef
}).sort_values("Ridge_Coef", key=np.abs, ascending=False)

print("\n=== Coefficient Comparison ===")
print(coef_df.head(15))

# 9) 계수 시각화 (상위 몇 개)
coef_df.set_index("Feature")[["Ridge_Coef","Lasso_Coef"]].head(20).plot(kind="bar", figsize=(12,5))
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Ridge vs Lasso Coefficients (Top 20 by absolute Ridge weight)")
plt.show()


#### 특성 개수를 자동 지정하기 위해 RFECV 사용

### RFECV, GridsearCV, StandardScaler를 사용한 모델 학습

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.feature_selection import RFECV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import clone  # ✅ 핵심: 서로 다른 인스턴스 보장

# ===============================
# 데이터 준비
# ===============================
path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path + 'life_expectancy.csv'
df = pd.read_csv(file)

drop_cols = ['Population', 'Hepatitis B', 'Total expenditure', 'Country', 'Year']
df = df.drop(columns=drop_cols)
df = df.fillna(df.median(numeric_only=True))

y = df["Life expectancy"]
X = df.drop(columns=["Life expectancy"])
X['Status'] = X['Status'].map({'Developing': 0, 'Developed': 1})
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cv5 = KFold(n_splits=5, shuffle=True, random_state=42)

def run_model(name, base_estimator, param_grid):
    print(f"\n===== {name} =====")

    # ✅ 서로 다른 인스턴스 사용 (클론)
    fs_est = clone(base_estimator)
    clf_est = clone(base_estimator)

    rfecv = RFECV(
        estimator=fs_est,
        step=1,
        cv=cv5,
        scoring="r2",
        n_jobs=1,                # ✅ 내부 병렬 비활성화 (안정화)
        min_features_to_select=1
    )

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("feature_sel", rfecv),
        ("clf", clf_est)
    ])

    grid = GridSearchCV(
        pipe,
        param_grid=param_grid,
        cv=cv5,
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )

    grid.fit(X_train, y_train)

    print("Best params:", grid.best_params_ if param_grid else "(no params)")
    print("Best CV R2:", f"{grid.best_score_:.4f}")

    sel = grid.best_estimator_.named_steps["feature_sel"]
    mask = sel.support_
    selected_feats = [f for f, keep in zip(feature_names, mask) if keep]
    print(f"Selected features: {sel.n_features_} / {len(feature_names)}")
    print("Selected feature names:", selected_feats)

    y_pred_train = grid.predict(X_train)
    y_pred_test = grid.predict(X_test)

    print("\n--- Train Report ---")
    print(f"R2:   {r2_score(y_train, y_pred_train):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}")
    print(f"MAE:  {mean_absolute_error(y_train, y_pred_train):.4f}")

    print("\n--- Test Report ---")
    print(f"R2:   {r2_score(y_test, y_pred_test):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
    print(f"MAE:  {mean_absolute_error(y_test, y_pred_test):.4f}")

# ===============================
# 1) LinearRegression (튜닝 없음)
# ===============================
run_model(
    "LinearRegression",
    LinearRegression(),
    param_grid={}
)

# ===============================
# 2) Ridge (alpha 동기화)
# ===============================
run_model(
    "Ridge",
    Ridge(),
    param_grid={
        "feature_sel__estimator__alpha": [0.1, 1, 10, 100],
        "clf__alpha": [0.1, 1, 10, 100],
    }
)

# ===============================
# 3) Lasso (alpha 동기화 + 수렴여유)
# ===============================
run_model(
    "Lasso",
    Lasso(max_iter=20000),
    param_grid={
        "feature_sel__estimator__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "clf__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
    }
)

# ===============================
# 4) ElasticNet (alpha/l1_ratio 동기화 + 수렴여유)
# ===============================
run_model(
    "ElasticNet",
    ElasticNet(max_iter=20000),
    param_grid={
        "feature_sel__estimator__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "clf__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "feature_sel__estimator__l1_ratio": [0.2, 0.5, 0.8],
        "clf__l1_ratio": [0.2, 0.5, 0.8],
    }
)


### RFECV, GridsearCV를 사용한 모델 학습

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.feature_selection import RFECV
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import clone  # ✅ 서로 다른 인스턴스 보장

# ===============================
# 데이터 준비
# ===============================
path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path + 'life_expectancy.csv'
df = pd.read_csv(file)

drop_cols = ['Population', 'Hepatitis B', 'Total expenditure', 'Country', 'Year']
df = df.drop(columns=drop_cols)
df = df.fillna(df.median(numeric_only=True))

y = df["Life expectancy"]
X = df.drop(columns=["Life expectancy"])
X['Status'] = X['Status'].map({'Developing': 0, 'Developed': 1})
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cv5 = KFold(n_splits=5, shuffle=True, random_state=42)

def run_model(name, base_estimator, param_grid):
    print(f"\n===== {name} =====")

    # ✅ 서로 다른 인스턴스 사용 (클론)
    fs_est = clone(base_estimator)
    clf_est = clone(base_estimator)

    rfecv = RFECV(
        estimator=fs_est,
        step=1,
        cv=cv5,
        scoring="r2",
        n_jobs=1,                # 내부 병렬 비활성화 (안정화)
        min_features_to_select=1
    )

    # 🔻 스케일링 제거
    pipe = Pipeline([
        ("feature_sel", rfecv),
        ("clf", clf_est)
    ])

    grid = GridSearchCV(
        pipe,
        param_grid=param_grid,
        cv=cv5,
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )

    grid.fit(X_train, y_train)

    print("Best params:", grid.best_params_ if param_grid else "(no params)")
    print("Best CV R2:", f"{grid.best_score_:.4f}")

    sel = grid.best_estimator_.named_steps["feature_sel"]
    mask = sel.support_
    selected_feats = [f for f, keep in zip(feature_names, mask) if keep]
    print(f"Selected features: {sel.n_features_} / {len(feature_names)}")
    print("Selected feature names:", selected_feats)

    y_pred_train = grid.predict(X_train)
    y_pred_test = grid.predict(X_test)

    print("\n--- Train Report ---")
    print(f"R2:   {r2_score(y_train, y_pred_train):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}")
    print(f"MAE:  {mean_absolute_error(y_train, y_pred_train):.4f}")

    print("\n--- Test Report ---")
    print(f"R2:   {r2_score(y_test, y_pred_test):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
    print(f"MAE:  {mean_absolute_error(y_test, y_pred_test):.4f}")

# ===============================
# 1) LinearRegression (튜닝 없음)
# ===============================
run_model(
    "LinearRegression",
    LinearRegression(),
    param_grid={}
)

# ===============================
# 2) Ridge (alpha 동기화)
# ===============================
run_model(
    "Ridge",
    Ridge(),
    param_grid={
        "feature_sel__estimator__alpha": [0.1, 1, 10, 100],
        "clf__alpha": [0.1, 1, 10, 100],
    }
)

# ===============================
# 3) Lasso (alpha 동기화 + 수렴여유)
# ===============================
run_model(
    "Lasso",
    Lasso(max_iter=20000),
    param_grid={
        "feature_sel__estimator__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "clf__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
    }
)

# ===============================
# 4) ElasticNet (alpha/l1_ratio 동기화 + 수렴여유)
# ===============================
run_model(
    "ElasticNet",
    ElasticNet(max_iter=20000),
    param_grid={
        "feature_sel__estimator__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "clf__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "feature_sel__estimator__l1_ratio": [0.2, 0.5, 0.8],
        "clf__l1_ratio": [0.2, 0.5, 0.8],
    }
)


#### RFECV 대신 RFE = 5 사용

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.feature_selection import RFE
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import clone

# ===============================
# 데이터 준비
# ===============================
path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path + 'life_expectancy.csv'
df = pd.read_csv(file)

drop_cols = ['Population', 'Hepatitis B', 'Total expenditure', 'Country', 'Year']
df = df.drop(columns=drop_cols)
df = df.fillna(df.median(numeric_only=True))

y = df["Life expectancy"]
X = df.drop(columns=["Life expectancy"])
X['Status'] = X['Status'].map({'Developing': 0, 'Developed': 1})
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cv5 = KFold(n_splits=5, shuffle=True, random_state=42)
MAX_K = 5 # ✅ 선택 특징 최대 개수 제한

def run_model(name, base_estimator, param_grid):
    print(f"\n===== {name} =====")

    fs_est = clone(base_estimator)
    clf_est = clone(base_estimator)

    # RFE로 교차검증 내에서 특징 개수(1..MAX_K) 튜닝
    rfe = RFE(estimator=fs_est, step=1)

    pipe = Pipeline([
        ("feature_sel", rfe),
        ("clf", clf_est)
    ])

    # 공통: n_features_to_select 를 1..MAX_K 범위로 탐색 (최대치 제한)
    base_grid = {
        "feature_sel__n_features_to_select": list(range(1, min(MAX_K, X_train.shape[1]) + 1))
    }
    # 모델별 그리드 병합
    full_grid = {**base_grid, **param_grid}

    grid = GridSearchCV(
        pipe,
        param_grid=full_grid,
        cv=cv5,
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )

    grid.fit(X_train, y_train)

    print("Best params:", grid.best_params_ if full_grid else "(no params)")
    print("Best CV R2:", f"{grid.best_score_:.4f}")

    sel = grid.best_estimator_.named_steps["feature_sel"]
    mask = sel.support_
    selected_feats = [f for f, keep in zip(feature_names, mask) if keep]
    print(f"Selected features: {sel.n_features_in_ if hasattr(sel,'n_features_in_') else 'NA'} total → {mask.sum()} used (≤ {MAX_K})")
    print("Selected feature names:", selected_feats)

    # 성능 평가
    y_pred_train = grid.predict(X_train)
    y_pred_test = grid.predict(X_test)

    print("\n--- Train Report ---")
    print(f"R2:   {r2_score(y_train, y_pred_train):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}")
    print(f"MAE:  {mean_absolute_error(y_train, y_pred_train):.4f}")

    print("\n--- Test Report ---")
    print(f"R2:   {r2_score(y_test, y_pred_test):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
    print(f"MAE:  {mean_absolute_error(y_test, y_pred_test):.4f}")

# ===============================
# 1) LinearRegression (튜닝 없음)
# ===============================
run_model(
    "LinearRegression",
    LinearRegression(),
    param_grid={}  # 특징 개수만 1..MAX_K로 튜닝
)

# ===============================
# 2) Ridge (alpha 동기화)
# ===============================
run_model(
    "Ridge",
    Ridge(),
    param_grid={
        "feature_sel__estimator__alpha": [0.1, 1, 10, 100],
        "clf__alpha": [0.1, 1, 10, 100],
    }
)

# ===============================
# 3) Lasso (alpha 동기화 + 수렴여유)
# ===============================
run_model(
    "Lasso",
    Lasso(max_iter=20000),
    param_grid={
        "feature_sel__estimator__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "clf__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
    }
)

# ===============================
# 4) ElasticNet (alpha/l1_ratio 동기화 + 수렴여유)
# ===============================
run_model(
    "ElasticNet",
    ElasticNet(max_iter=20000),
    param_grid={
        "feature_sel__estimator__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "clf__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "feature_sel__estimator__l1_ratio": [0.2, 0.5, 0.8],
        "clf__l1_ratio": [0.2, 0.5, 0.8],
    }
)


#### RFECV 대신 RFE = 5 사용 & 스케일링 추가

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.feature_selection import RFE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import clone

# ===============================
# 데이터 준비
# ===============================
path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path + 'life_expectancy.csv'
df = pd.read_csv(file)

drop_cols = ['Population', 'Hepatitis B', 'Total expenditure', 'Country', 'Year']
df = df.drop(columns=drop_cols)
df = df.fillna(df.median(numeric_only=True))

y = df["Life expectancy"]
X = df.drop(columns=["Life expectancy"])
X['Status'] = X['Status'].map({'Developing': 0, 'Developed': 1})
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cv5 = KFold(n_splits=5, shuffle=True, random_state=42)
MAX_K = 5  # ✅ 선택 특징 최대 개수 제한

def run_model(name, base_estimator, param_grid):
    print(f"\n===== {name} =====")

    fs_est = clone(base_estimator)
    clf_est = clone(base_estimator)

    # RFE (특징 개수는 그리드에서 1..MAX_K 탐색)
    rfe = RFE(estimator=fs_est, step=1)

    # ✅ 스케일링 추가 (누수 방지: 파이프라인 내부에서 CV에 포함)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("feature_sel", rfe),
        ("clf", clf_est)
    ])

    # 공통: n_features_to_select 를 1..MAX_K 범위로 탐색
    base_grid = {
        "feature_sel__n_features_to_select": list(range(1, min(MAX_K, X_train.shape[1]) + 1))
    }
    full_grid = {**base_grid, **param_grid}

    grid = GridSearchCV(
        pipe,
        param_grid=full_grid,
        cv=cv5,
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )

    grid.fit(X_train, y_train)

    print("Best params:", grid.best_params_ if full_grid else "(no params)")
    print("Best CV R2:", f"{grid.best_score_:.4f}")

    sel = grid.best_estimator_.named_steps["feature_sel"]
    mask = sel.support_
    selected_feats = [f for f, keep in zip(feature_names, mask) if keep]
    print(f"Selected features: {mask.sum()} used (≤ {MAX_K})")
    print("Selected feature names:", selected_feats)

    # 성능 평가
    y_pred_train = grid.predict(X_train)
    y_pred_test = grid.predict(X_test)

    print("\n--- Train Report ---")
    print(f"R2:   {r2_score(y_train, y_pred_train):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}")
    print(f"MAE:  {mean_absolute_error(y_train, y_pred_train):.4f}")

    print("\n--- Test Report ---")
    print(f"R2:   {r2_score(y_test, y_pred_test):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
    print(f"MAE:  {mean_absolute_error(y_test, y_pred_test):.4f}")

# ===============================
# 1) LinearRegression (튜닝 없음)
# ===============================
run_model(
    "LinearRegression",
    LinearRegression(),
    param_grid={}
)

# ===============================
# 2) Ridge (alpha 동기화)
# ===============================
run_model(
    "Ridge",
    Ridge(),
    param_grid={
        "feature_sel__estimator__alpha": [0.1, 1, 10, 100],
        "clf__alpha": [0.1, 1, 10, 100],
    }
)

# ===============================
# 3) Lasso (alpha 동기화 + 수렴여유)
# ===============================
run_model(
    "Lasso",
    Lasso(max_iter=20000),
    param_grid={
        "feature_sel__estimator__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "clf__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
    }
)

# ===============================
# 4) ElasticNet (alpha/l1_ratio 동기화 + 수렴여유)
# ===============================
run_model(
    "ElasticNet",
    ElasticNet(max_iter=20000),
    param_grid={
        "feature_sel__estimator__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "clf__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "feature_sel__estimator__l1_ratio": [0.2, 0.5, 0.8],
        "clf__l1_ratio": [0.2, 0.5, 0.8],
    }
)


#### 점수가 기존보다 낮게 나옴, 데이터 전처리 문제
#### 다른 특성들을 모두 최대한 이용

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.feature_selection import RFECV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import clone  # ✅ 핵심: 서로 다른 인스턴스 보장

# ===============================
# 데이터 준비
# ===============================
path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path + 'life_expectancy.csv'
df = pd.read_csv(file)

drop_cols = ['Year', 'Country']
df = df.drop(columns=drop_cols)
df.dropna(inplace=True)

y = df["Life expectancy"]
X = df.drop(columns=["Life expectancy"])
X['Status'] = X['Status'].map({'Developing': 0, 'Developed': 1})
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cv5 = KFold(n_splits=5, shuffle=True, random_state=42)

def run_model(name, base_estimator, param_grid):
    print(f"\n===== {name} =====")

    # ✅ 서로 다른 인스턴스 사용 (클론)
    fs_est = clone(base_estimator)
    clf_est = clone(base_estimator)

    rfecv = RFECV(
        estimator=fs_est,
        step=1,
        cv=cv5,
        scoring="r2",
        n_jobs=1,                # ✅ 내부 병렬 비활성화 (안정화)
        min_features_to_select=1
    )

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("feature_sel", rfecv),
        ("clf", clf_est)
    ])

    grid = GridSearchCV(
        pipe,
        param_grid=param_grid,
        cv=cv5,
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )

    grid.fit(X_train, y_train)

    print("Best params:", grid.best_params_ if param_grid else "(no params)")
    print("Best CV R2:", f"{grid.best_score_:.4f}")

    sel = grid.best_estimator_.named_steps["feature_sel"]
    mask = sel.support_
    selected_feats = [f for f, keep in zip(feature_names, mask) if keep]
    print(f"Selected features: {sel.n_features_} / {len(feature_names)}")
    print("Selected feature names:", selected_feats)

    y_pred_train = grid.predict(X_train)
    y_pred_test = grid.predict(X_test)

    print("\n--- Train Report ---")
    print(f"R2:   {r2_score(y_train, y_pred_train):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}")
    print(f"MAE:  {mean_absolute_error(y_train, y_pred_train):.4f}")

    print("\n--- Test Report ---")
    print(f"R2:   {r2_score(y_test, y_pred_test):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
    print(f"MAE:  {mean_absolute_error(y_test, y_pred_test):.4f}")

# ===============================
# 1) LinearRegression (튜닝 없음)
# ===============================
run_model(
    "LinearRegression",
    LinearRegression(),
    param_grid={}
)

# ===============================
# 2) Ridge (alpha 동기화)
# ===============================
run_model(
    "Ridge",
    Ridge(),
    param_grid={
        "feature_sel__estimator__alpha": [0.1, 1, 10, 100],
        "clf__alpha": [0.1, 1, 10, 100],
    }
)

# ===============================
# 3) Lasso (alpha 동기화 + 수렴여유)
# ===============================
run_model(
    "Lasso",
    Lasso(max_iter=20000),
    param_grid={
        "feature_sel__estimator__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "clf__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
    }
)

# ===============================
# 4) ElasticNet (alpha/l1_ratio 동기화 + 수렴여유)
# ===============================
run_model(
    "ElasticNet",
    ElasticNet(max_iter=20000),
    param_grid={
        "feature_sel__estimator__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "clf__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "feature_sel__estimator__l1_ratio": [0.2, 0.5, 0.8],
        "clf__l1_ratio": [0.2, 0.5, 0.8],
    }
)


#### 스케일링 제거

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.feature_selection import RFECV
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.base import clone  # ✅ 서로 다른 인스턴스 보장

# ===============================
# 데이터 준비
# ===============================
path = 'https://github.com/dongupak/DataML/raw/main/csv/'
file = path + 'life_expectancy.csv'
df = pd.read_csv(file)

drop_cols = ['Year', 'Country']
df = df.drop(columns=drop_cols)
df.dropna(inplace=True)

y = df["Life expectancy"]
X = df.drop(columns=["Life expectancy"])
X['Status'] = X['Status'].map({'Developing': 0, 'Developed': 1})
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cv5 = KFold(n_splits=5, shuffle=True, random_state=42)

def run_model(name, base_estimator, param_grid):
    print(f"\n===== {name} =====")

    # ✅ 서로 다른 인스턴스 사용 (클론)
    fs_est = clone(base_estimator)
    clf_est = clone(base_estimator)

    rfecv = RFECV(
        estimator=fs_est,
        step=1,
        cv=cv5,
        scoring="r2",
        n_jobs=1,
        min_features_to_select=1
    )

    # 🔻 스케일링 제거
    pipe = Pipeline([
        ("feature_sel", rfecv),
        ("clf", clf_est)
    ])

    grid = GridSearchCV(
        pipe,
        param_grid=param_grid,
        cv=cv5,
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )

    grid.fit(X_train, y_train)

    print("Best params:", grid.best_params_ if param_grid else "(no params)")
    print("Best CV R2:", f"{grid.best_score_:.4f}")

    sel = grid.best_estimator_.named_steps["feature_sel"]
    mask = sel.support_
    selected_feats = [f for f, keep in zip(feature_names, mask) if keep]
    print(f"Selected features: {sel.n_features_} / {len(feature_names)}")
    print("Selected feature names:", selected_feats)

    y_pred_train = grid.predict(X_train)
    y_pred_test = grid.predict(X_test)

    print("\n--- Train Report ---")
    print(f"R2:   {r2_score(y_train, y_pred_train):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}")
    print(f"MAE:  {mean_absolute_error(y_train, y_pred_train):.4f}")

    print("\n--- Test Report ---")
    print(f"R2:   {r2_score(y_test, y_pred_test):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
    print(f"MAE:  {mean_absolute_error(y_test, y_pred_test):.4f}")

# ===============================
# 1) LinearRegression (튜닝 없음)
# ===============================
run_model(
    "LinearRegression",
    LinearRegression(),
    param_grid={}
)

# ===============================
# 2) Ridge
# ===============================
run_model(
    "Ridge",
    Ridge(),
    param_grid={
        "feature_sel__estimator__alpha": [0.1, 1, 10, 100],
        "clf__alpha": [0.1, 1, 10, 100],
    }
)

# ===============================
# 3) Lasso
# ===============================
run_model(
    "Lasso",
    Lasso(max_iter=20000),
    param_grid={
        "feature_sel__estimator__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "clf__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
    }
)

# ===============================
# 4) ElasticNet
# ===============================
run_model(
    "ElasticNet",
    ElasticNet(max_iter=20000),
    param_grid={
        "feature_sel__estimator__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "clf__alpha": [0.0005, 0.001, 0.01, 0.1, 1],
        "feature_sel__estimator__l1_ratio": [0.2, 0.5, 0.8],
        "clf__l1_ratio": [0.2, 0.5, 0.8],
    }
)
